In [1]:
%reload_ext autotime
import pandas as pd
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
from pprint import pprint
import json5 as json # This is a more forgiving JSON parser that can handle comments, single quotes, and trailing commas
from glob import glob
from tqdm import tqdm
from openai import OpenAI
import base64
files = sorted(glob("../supplements_videos/*.json"))
print(len(files))

6871
time: 854 ms (started: 2026-06-22 09:08:11 +12:00)


In [2]:
from glob import glob
pd.Series([os.path.splitext(f)[1] for f in glob("../supplements_videos/*")]).value_counts()

.json    6871
.mp4     4689
.webm    2078
.mp3       72
.m4a       30
.mkv        9
.part       4
Name: count, dtype: int64

time: 61.2 ms (started: 2026-06-22 09:08:12 +12:00)


In [3]:
print(pd.to_timedelta("00:" + pd.read_csv("../data/supplements.csv").duration).describe())
print("Total video time:", pd.to_timedelta("00:" + pd.read_csv("../data/supplements.csv").duration).sum())

count                     10998
mean     0 days 00:00:59.010183
std      0 days 00:00:40.011057
min             0 days 00:00:01
25%             0 days 00:00:28
50%             0 days 00:00:53
75%             0 days 00:01:21
max             0 days 00:03:00
Name: duration, dtype: object
Total video time: 7 days 12:16:34
time: 118 ms (started: 2026-06-22 09:08:12 +12:00)


In [4]:
pd.read_csv("../data/supplements.csv").sort_values("duration")

,link,duration,title,source,author
2586,https://www.instagram.com/p/DYhBrzBjCNs/,0:01,Hot flashes can be an unpleasant aspect of the menopause ...,Instagram,Tweak India
2364,https://www.instagram.com/p/Cvwl00wtZ3r/,0:02,Menopause diet 101 Make sure you're eating enough protein ...,Instagram,Nutritionist | 30:30:30 method
4853,https://www.instagram.com/reel/DY7iaRBxCcP/,0:02,Natalie Yco | Hi Fam! 🙋🏾‍♀️ Tell me your holy grail ...,Instagram,Natalie Yco
2773,https://www.instagram.com/reel/Cvwl00wtZ3r/,0:02,Menopause diet 101 Make sure you're eating enough protein ...,Instagram,Nutritionist | 30:30:30 method
3744,https://www.instagram.com/reel/DU2jYyyCK0L/,0:02,HAIR FOOD | MENO is a menopause specific food ...,Instagram,Oxygen Hair Beaconsfield
...,...,...,...,...,...
2749,https://www.instagram.com/reel/C_vPjPJsZy7/?hl=en,3:00,“Meno belly” describes the increase in abdominal fat that often ...,Instagram,LUCY - 2 Million + YouTube
10704,https://www.youtube.com/shorts/zZ7cScqJX_8,3:00,Oral Progesterone for Sleep in Perimenopause and Menopause,YouTube,"Jennifer Roelands, MD Perimenopause, Longevity"
9254,https://www.youtube.com/shorts/D0LGbGhP7RU,3:00,What Is Perimenopause? Dr. Mary Claire Haver Explains ...,YouTube,Katie Couric
5676,https://www.instagram.com/reel/DZle9U6pFxT/,3:00,"on Instagram: ""LADIES ‼️ LET'S TALK PERIMENOPAUSE ...",Instagram,Brooke | Motherhood | Mindset | Growth |


time: 78.8 ms (started: 2026-06-22 09:08:13 +12:00)


In [17]:
import ffmpeg

def has_audio(video_path):
    try:
        # Probe the video file specifically selecting audio streams
        probe = ffmpeg.probe(video_path, select_streams='a')
        # If the 'streams' list is not empty, an audio track exists
        return len(probe['streams']) > 0
    except ffmpeg.Error as e:
        print(f"Error checking file: {e}")
        return False

time: 2.33 ms (started: 2026-06-22 09:13:33 +12:00)


In [6]:
json_filename = "../supplements_videos/WTF is happening to my body？ #perimenopause #womenshealth #menopause  [7652060389173234952].info.json"
with open(json_filename) as f:
    data = json.load(f)
display(pd.json_normalize(data))
print(data["description"])

,id,formats,channel,channel_id,uploader,uploader_id,channel_url,uploader_url,track,artists,duration,title,description,timestamp,view_count,like_count,repost_count,comment_count,save_count,thumbnails,webpage_url,webpage_url_basename,webpage_url_domain,extractor,extractor_key,thumbnail,display_id,fulltitle,duration_string,upload_date,artist,epoch,ext,vcodec,acodec,format_id,tbr,quality,preference,filesize,width,height,url,protocol,video_ext,audio_ext,resolution,dynamic_range,aspect_ratio,cookies,format,_type,http_headers.User-Agent,http_headers.Accept,http_headers.Accept-Language,http_headers.Sec-Fetch-Mode,http_headers.Referer,_version.version,_version.release_git_head,_version.repository
0,7652060389173234952,"[{'ext': 'mp4', 'vcodec': 'h264', 'acodec': 'aac', 'format_id': 'h264_540p_351946-0', 'tbr': 351...",Sasha Exeter,MS4wLjABAAAAPRTFMxdwow4LaD81UduslWxKFhP2tKtyDTkLjzdq7koq57nWZsI8q5jagsGN80sD,sashaexeter,6764696465399907333,https://www.tiktok.com/@MS4wLjABAAAAPRTFMxdwow4LaD81UduslWxKFhP2tKtyDTkLjzdq7koq57nWZsI8q5jagsGN...,https://www.tiktok.com/@sashaexeter,original sound,[Sasha Exeter],147,WTF is happening to my body? #perimenopause #womenshealth #menopause,WTF is happening to my body? #perimenopause #womenshealth #menopause,1781634165,2254,170,12,36,14,"[{'id': 'dynamicCover', 'url': 'https://p16-common-sign.tiktokcdn.com/tos-alisg-p-0037/oYL1GABME...",https://www.tiktok.com/@sashaexeter/video/7652060389173234952,7652060389173234952,tiktok.com,TikTok,TikTok,https://p16-common-sign.tiktokcdn.com/tos-alisg-p-0037/okx1ESAQiAUcGt0jBYB2aoliIADBKFopPXIBE~tpl...,7652060389173234952,WTF is happening to my body? #perimenopause #womenshealth #menopause,2:27,20260616,Sasha Exeter,1781836603,mp4,h265,aac,bytevc1_720p_648975-1,648,2,-1,6802594,720,1280,https://v19-webapp-prime.tiktok.com/video/tos/alisg/tos-alisg-pv-0037/o4APAUBoQxBgEYXoIAAStiH0EA...,https,mp4,none,720x1280,SDR,0.56,ttwid=1%7CQYqqQ0c0gBpHMLEVroSuf2ReGqV65VYTzGQnx_wOwdQ%7C1781836154%7Ccb7ea7b6498aae36aeb8d3b0bf7...,bytevc1_720p_648975-1 - 720x1280,video,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0....","text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8","en-us,en;q=0.5",navigate,https://www.tiktok.com/@sashaexeter/video/7652060389173234952,2026.03.17,04d6974f502bbdfaed72c624344f262e30ad9708,yt-dlp/yt-dlp


WTF is happening to my body? #perimenopause #womenshealth #menopause 
time: 467 ms (started: 2026-06-22 09:08:14 +12:00)


In [7]:
def get_prompt(data):
    return f"""This is a video downloaded from {data['extractor']}. Here's the description of the video: {data['description']}.

        The creator of the video is {data.get('channel', 'an unknown channel')} ({data.get('uploader', 'an unknown uploader')})
        It has {data.get('like_count', 'an unknown number of')} likes, {data.get('view_count', 'an unknown number of')} views, and {data.get('comment_count', 'an unknown number of')} comments.
        Taking into account this description, and the video, extract the following information, in JSON format:
        description: What is happening in the video? Provide a detailed description of the actions, context, and any notable elements present in the video.
        transcript: If there is any spoken content in the video, transcribe it into English. If there is no spoken content, indicate "No spoken content".
        tone: What is the overall tone or mood of the video? Is it humorous, serious, educational, emotional, etc.?
        supplements: Does the video mention any supplements, vitamins, or medications? If so, list them. If not, indicate "No supplements mentioned".
        active_ingredients: If any supplements are mentioned, list the active ingredients in those supplements. If no supplements are mentioned, indicate "No active ingredients mentioned".
        symptoms: Does the video mention any specific symptoms, conditions, or health issues? If so, list them. If not, indicate "No symptoms mentioned".
        menopause: Is the video specifically targeting the supplement towards menopause-related symptoms or conditions? Answer True or False.
        language: What language is this video in?
        marketing: Is this video promoting or advertising any product, service, brand, or organization? If so, what is it? Otherwise, indicate "No marketing content".
        job: For the main speaker, what is their job or profession? If it is not mentioned in the video, indicate "No job information". A comma separated string, one or more of the following: therapist, psychologist, pediatrician, doctor, nurse, teacher, professor, social worker, counselor, coach, influencer, content creator?
        sentiment: Does this video recommend a particular supplement, discourage it, or is it neutral? One of negative, neutral or positive
        criticism: If the video is critical of a particular supplement, what are the main criticisms mentioned? 
        alternative_strategies: Does the video mention any alternative strategies to supplements? If so, what are they? A comma separated string. If no alternatives are mentioned, indicate "No alternative strategies mentioned".
        usefulness: Rate the overall usefulness of the video on a scale from 1 to 10, where 1 is not useful at all and 10 is extremely useful.
        misleading: Rate the extent to which the video contains misleading or inaccurate information on a scale from 1 to 10, where 1 is not misleading at all and 10 is extremely misleading.
        quality: Rate the overall quality of the video on a scale from 1 to 10, where 1 is very poor quality and 10 is excellent quality.
        personal_experience: Does the speaker mention any personal experience with supplements? If so, briefly summarize it.

        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid
    """
print(get_prompt(data))

This is a video downloaded from TikTok. Here's the description of the video: WTF is happening to my body? #perimenopause #womenshealth #menopause .

        The creator of the video is Sasha Exeter (sashaexeter)
        It has 170 likes, 2254 views, and 36 comments.
        Taking into account this description, and the video, extract the following information, in JSON format:
        description: What is happening in the video? Provide a detailed description of the actions, context, and any notable elements present in the video.
        transcript: If there is any spoken content in the video, transcribe it into English. If there is no spoken content, indicate "No spoken content".
        tone: What is the overall tone or mood of the video? Is it humorous, serious, educational, emotional, etc.?
        supplements: Does the video mention any supplements, vitamins, or medications? If so, list them. If not, indicate "No supplements mentioned".
        active_ingredients: If any supplement

In [9]:
client = OpenAI(base_url="https://ai.cer-sandbox.cloud.edu.au/v1", api_key="not needed")

video_filename = json_filename.replace("info.json", data["ext"])
print(video_filename)
with open(video_filename, "rb") as video_file:
    base64_video = "data:video/" + data["ext"] + ";base64," + base64.b64encode(video_file.read()).decode("utf-8")
print(len(base64_video))

reasoning_budget = 16384
grace_period = 1024

response = client.chat.completions.create(
    model="nemotron_3_nano_omni",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": base64_video}},
                {"type": "text", "text": get_prompt(data)},
            ],
        }
    ],
    max_tokens=20480,
    temperature=0.6,
    top_p=0.95,
    extra_body={
        "thinking_token_budget": reasoning_budget + grace_period,
        "chat_template_kwargs": {
            "enable_thinking": True,
            "reasoning_budget": reasoning_budget,
        },
        "mm_processor_kwargs": {"use_audio_in_video": has_audio(video_filename)},
    },
)
print(response.choices[0].message.reasoning)
pprint(json.loads(response.choices[0].message.content))

../supplements_videos/WTF is happening to my body？ #perimenopause #womenshealth #menopause  [7652060389173234952].mp4
9070150
We need to extract info from video. The video is a woman talking about perimenopause, brain fog, etc. Let's parse.

Description: She is sitting on a couch, wearing workout gear, talking directly to camera. She mentions "WTF is happening to my body?" and "#perimenopause #womenshealth #menopause". She talks about brain fog, losing train of thought, forgetting what she was trying to say. She mentions symptoms like brain fog. She says it's also a symptom. She mentions that she lost her train of thought. She also mentions maybe other symptoms? Let's see. She mentions "brain fog is also a symptom". She doesn't mention specific supplements, vitamins, or meds. She might talk about something else but likely not.

Transcript: Need to transcribe spoken content. Let's attempt to capture main sentences.

She says: "Oh...and brainfog is also a symptom. Literally just lost my 